# Scouting Signals

This notebook builds deterministic scouting outputs from the clean gold datasets created by `01_data_processing.ipynb`.

The goal is to add scout-facing information that does not require heavy model training:

- player trend signals
- consistency and volatility scores
- short-term expected/floor/ceiling ranges
- replacement candidate ranking

All tables are derived from point-in-time or season-level gold data to keep the design consistent with the main data-processing notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

## Data Paths

The path resolver mirrors the gold-data convention used in `01_data_processing.ipynb`: prefer the Colab Drive data folder, fall back to local project `data`, and allow `NBA_SCOUT_DATA_DIR` to override both.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
COLAB_DRIVE_DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
COLAB_LEGACY_DRIVE_DATA_DIR = Path("/content/drive/My Drive/nba-scout-assistant/data")
LOCAL_DATA_DIR = PROJECT_ROOT / "data"

REQUIRED_GOLD_FILES = [
    Path("gold/performance_training_clean.parquet"),
    Path("gold/player_role_features_clean.parquet"),
    Path("gold/salary_training_clean.parquet"),
]


def running_in_colab() -> bool:
    """Input: runtime environment. Output: True when the notebook is running in Google Colab."""
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def mount_google_drive_if_available() -> None:
    """Input: Colab runtime. Output: mounted Google Drive when the notebook is running in Colab."""
    if not running_in_colab():
        return
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")


def has_required_gold_files(data_dir: Path) -> bool:
    """Input: candidate data directory. Output: whether all required gold files exist."""
    return all((data_dir / relative_path).exists() for relative_path in REQUIRED_GOLD_FILES)


def missing_required_gold_files(data_dir: Path) -> list[str]:
    """Input: candidate data directory. Output: missing required gold file paths."""
    return [str(data_dir / relative_path) for relative_path in REQUIRED_GOLD_FILES if not (data_dir / relative_path).exists()]


def resolve_data_dir() -> Path:
    """Input: environment/default locations. Output: data directory that contains the required gold files."""
    mount_google_drive_if_available()

    env_data_dir = os.getenv("NBA_SCOUT_DATA_DIR")
    if env_data_dir:
        candidate = Path(env_data_dir).expanduser().resolve()
        if has_required_gold_files(candidate):
            return candidate
        missing = "\n".join(missing_required_gold_files(candidate))
        raise FileNotFoundError(
            "NBA_SCOUT_DATA_DIR is set, but required gold files are missing:\n"
            f"{missing}"
        )

    candidates = [COLAB_DRIVE_DATA_DIR, COLAB_LEGACY_DRIVE_DATA_DIR, LOCAL_DATA_DIR]
    for candidate in candidates:
        if has_required_gold_files(candidate):
            return candidate.resolve()

    checked = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the required gold data files. Checked data directories:\n"
        f"{checked}\n\n"
        "In Colab, make sure Drive is mounted and the data folder exists at:\n"
        f"{COLAB_DRIVE_DATA_DIR}\n"
        "Or set NBA_SCOUT_DATA_DIR to the folder that contains the gold directory."
    )


DATA_DIR = resolve_data_dir()
GOLD_DIR = DATA_DIR / "gold"

PERFORMANCE_PATH = GOLD_DIR / "performance_training_clean.parquet"
ROLE_PATH = GOLD_DIR / "player_role_features_clean.parquet"
SALARY_PATH = GOLD_DIR / "salary_training_clean.parquet"

TREND_SIGNAL_PATH = GOLD_DIR / "player_trend_signals.parquet"
CONSISTENCY_SIGNAL_PATH = GOLD_DIR / "player_consistency_signals.parquet"
FLOOR_CEILING_SIGNAL_PATH = GOLD_DIR / "short_term_floor_ceiling_signals.parquet"
FLOOR_CEILING_EVAL_PATH = GOLD_DIR / "short_term_floor_ceiling_evaluation.parquet"
REPLACEMENT_EXAMPLES_PATH = GOLD_DIR / "replacement_candidate_examples.parquet"

print("DATA_DIR:", DATA_DIR)
for path in [PERFORMANCE_PATH, ROLE_PATH, SALARY_PATH]:
    print(path.name, "exists=", path.exists())

In [ ]:
performance = pd.read_parquet(PERFORMANCE_PATH)
role_features = pd.read_parquet(ROLE_PATH)
salary = pd.read_parquet(SALARY_PATH)

performance["as_of_date"] = pd.to_datetime(performance["as_of_date"])
performance = performance.sort_values(["player_id", "season", "as_of_date", "game_id"]).reset_index(drop=True)
role_features = role_features.sort_values(["season", "player_id"]).reset_index(drop=True)
salary = salary.sort_values(["season_start_year", "player_id"]).reset_index(drop=True)

print("performance", performance.shape)
print("role_features", role_features.shape)
print("salary", salary.shape)

## Shared Helpers

These helpers keep the signal definitions explicit and reusable. They avoid model fitting and only use already-cleaned columns.

In [ ]:
STAT_CONFIG = {
    "pts": {"actual": "pts", "last_5": "pts_last_5", "last_10": "pts_last_10", "season_avg": "pts_season_avg", "target": "target_next_5_pts_avg"},
    "ast": {"actual": "ast", "last_5": "ast_last_5", "last_10": "ast_last_10", "season_avg": "ast_season_avg", "target": "target_next_5_ast_avg"},
    "reb": {"actual": "reb", "last_5": "reb_last_5", "last_10": "reb_last_10", "season_avg": "reb_season_avg", "target": "target_next_5_reb_avg"},
}


def season_start_year(season: object) -> int:
    """Input: NBA season label such as '2023-24'. Output: start year as integer."""
    return int(str(season)[:4])


def classify_direction(delta: pd.Series, tolerance: float) -> pd.Series:
    """Input: numeric delta and tolerance. Output: improving/stable/declining label."""
    return pd.Series(
        np.select([delta > tolerance, delta < -tolerance], ["improving", "declining"], default="stable"),
        index=delta.index,
    )


def robust_zscore(values: pd.Series) -> pd.Series:
    """Input: numeric series. Output: robust z-score using median absolute deviation."""
    numeric = pd.to_numeric(values, errors="coerce")
    median = numeric.median()
    mad = (numeric - median).abs().median()
    if pd.isna(mad) or mad == 0:
        std = numeric.std(ddof=0)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(numeric)), index=numeric.index)
        return (numeric - numeric.mean()) / std
    return 0.6745 * (numeric - median) / mad

## Player Trend Signals

Trend signals compare recent production against the player's season baseline and short-term baseline. This is designed for scouting interpretation, not as a supervised prediction target.

In [ ]:
def build_player_trend_signals(performance_df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean point-in-time performance table. Output: latest player-season trend labels and numeric deltas."""
    latest = (
        performance_df
        .sort_values(["player_id", "season", "as_of_date", "game_id"])
        .groupby(["player_id", "season"], as_index=False)
        .tail(1)
        .copy()
    )

    latest["season_start_year"] = latest["season"].map(season_start_year)
    latest["pts_recent_delta"] = latest["pts_last_5"] - latest["pts_season_avg"]
    latest["ast_recent_delta"] = latest["ast_last_5"] - latest["ast_season_avg"]
    latest["reb_recent_delta"] = latest["reb_last_5"] - latest["reb_season_avg"]
    latest["min_recent_delta"] = latest["min_last_5"] - latest["min_season_avg"]

    latest["pts_trend"] = classify_direction(latest["pts_recent_delta"], tolerance=1.5)
    latest["ast_trend"] = classify_direction(latest["ast_recent_delta"], tolerance=0.6)
    latest["reb_trend"] = classify_direction(latest["reb_recent_delta"], tolerance=0.8)
    latest["minutes_trend"] = classify_direction(latest["min_recent_delta"], tolerance=3.0)

    score_map = {"improving": 1, "stable": 0, "declining": -1}
    latest["production_trend_score"] = latest["pts_trend"].map(score_map) + latest["ast_trend"].map(score_map) + latest["reb_trend"].map(score_map)
    latest["overall_trend"] = classify_direction(latest["production_trend_score"], tolerance=0.5)

    output_cols = [
        "player_id", "season", "season_start_year", "team_id", "as_of_date",
        "pts_recent_delta", "ast_recent_delta", "reb_recent_delta", "min_recent_delta",
        "pts_trend", "ast_trend", "reb_trend", "minutes_trend",
        "production_trend_score", "overall_trend",
    ]
    return latest[output_cols].reset_index(drop=True)


player_trend_signals = build_player_trend_signals(performance)
player_trend_signals.to_parquet(TREND_SIGNAL_PATH, index=False)
print(f"Saved {TREND_SIGNAL_PATH} with shape {player_trend_signals.shape}")
display(player_trend_signals.tail(10))

## Trend Threshold Calibration

Trend thresholds are heuristic unless they are checked against validation behavior. This section evaluates candidate tolerances by comparing the current recent-vs-season delta with the future next-five-game delta.

The selected threshold for each stat is chosen on the validation split using directional hit rate among non-stable labels, with a minimum non-stable coverage requirement to avoid thresholds that only classify a tiny number of rows.

In [ ]:
TREND_CANDIDATE_THRESHOLDS = {
    "pts": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0],
    "ast": [0.2, 0.4, 0.6, 0.8, 1.0],
    "reb": [0.3, 0.5, 0.8, 1.0, 1.2],
    "min": [1.0, 2.0, 3.0, 4.0, 5.0],
}

TREND_EVAL_CONFIG = {
    "pts": {"delta_col": "pts_last_5_minus_season_avg", "target_col": "target_next_5_pts_avg", "baseline_col": "pts_season_avg"},
    "ast": {"delta_col": "ast_last_5_minus_season_avg", "target_col": "target_next_5_ast_avg", "baseline_col": "ast_season_avg"},
    "reb": {"delta_col": "reb_last_5_minus_season_avg", "target_col": "target_next_5_reb_avg", "baseline_col": "reb_season_avg"},
    "min": {"delta_col": "min_last_5_minus_season_avg", "target_col": None, "baseline_col": None},
}


def evaluate_trend_thresholds(df: pd.DataFrame, split_name: str = "validation", min_non_stable_rate: float = 0.10) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Input: performance table and candidate thresholds. Output: threshold metrics and selected validation thresholds."""
    rows = []
    eval_df = df[df["split"].eq(split_name)].copy()
    for stat, thresholds in TREND_CANDIDATE_THRESHOLDS.items():
        stat_config = TREND_EVAL_CONFIG[stat]
        delta_col = stat_config["delta_col"]
        if delta_col not in eval_df.columns:
            continue
        if stat_config["target_col"] is None:
            for tolerance in thresholds:
                labels = classify_direction(eval_df[delta_col], tolerance)
                rows.append({
                    "stat": stat,
                    "split": split_name,
                    "tolerance": tolerance,
                    "rows": len(eval_df),
                    "non_stable_rate": labels.ne("stable").mean(),
                    "directional_hit_rate": np.nan,
                    "future_delta_mean_improving": np.nan,
                    "future_delta_mean_stable": np.nan,
                    "future_delta_mean_declining": np.nan,
                    "selection_eligible": labels.ne("stable").mean() >= min_non_stable_rate,
                })
            continue

        target_col = stat_config["target_col"]
        baseline_col = stat_config["baseline_col"]
        valid = eval_df[[delta_col, target_col, baseline_col]].dropna().copy()
        valid["future_delta"] = valid[target_col] - valid[baseline_col]
        for tolerance in thresholds:
            labels = classify_direction(valid[delta_col], tolerance)
            non_stable = labels.ne("stable")
            predicted_direction = labels.map({"improving": 1, "declining": -1, "stable": 0}).to_numpy(dtype="int64")
            actual_direction = np.sign(valid["future_delta"].to_numpy(dtype="float64")).astype("int64")
            hit_rate = (predicted_direction[non_stable] == actual_direction[non_stable]).mean() if non_stable.any() else np.nan
            labeled = valid.assign(label=labels)
            future_means = labeled.groupby("label")["future_delta"].mean().to_dict()
            rows.append({
                "stat": stat,
                "split": split_name,
                "tolerance": tolerance,
                "rows": len(valid),
                "non_stable_rate": non_stable.mean(),
                "directional_hit_rate": hit_rate,
                "future_delta_mean_improving": future_means.get("improving", np.nan),
                "future_delta_mean_stable": future_means.get("stable", np.nan),
                "future_delta_mean_declining": future_means.get("declining", np.nan),
                "selection_eligible": non_stable.mean() >= min_non_stable_rate,
            })
    metrics = pd.DataFrame(rows)
    selected_rows = []
    for stat, group in metrics.groupby("stat"):
        eligible = group[group["selection_eligible"]].copy()
        if eligible.empty:
            eligible = group.copy()
        selected = eligible.sort_values(["directional_hit_rate", "non_stable_rate"], ascending=[False, False]).head(1)
        selected_rows.append(selected)
    selected = pd.concat(selected_rows, ignore_index=True) if selected_rows else pd.DataFrame()
    return metrics.sort_values(["stat", "tolerance"]).reset_index(drop=True), selected.sort_values("stat").reset_index(drop=True)


trend_threshold_evaluation, selected_trend_thresholds = evaluate_trend_thresholds(performance, split_name="validation")
print("Selected trend thresholds by validation:")
display(selected_trend_thresholds)
print("Trend threshold sensitivity:")
display(trend_threshold_evaluation)

## Consistency And Volatility Signals

Consistency is computed from actual game-level production within each player-season. The table avoids labels and model fitting; it summarizes how stable each player's scoring, playmaking, rebounding, and minutes were during a season.

In [ ]:
def build_player_consistency_signals(performance_df: pd.DataFrame, min_games: int = 10) -> pd.DataFrame:
    """Input: clean game-level performance table. Output: player-season consistency, volatility, and floor/ceiling descriptors."""
    rows = []
    for keys, group in performance_df.groupby(["player_id", "season", "team_id"], dropna=False):
        player_id, season, team_id = keys
        if len(group) < min_games:
            continue
        row = {"player_id": player_id, "season": season, "season_start_year": season_start_year(season), "team_id": team_id, "games_observed": len(group)}
        for stat in ["pts", "ast", "reb", "min"]:
            values = pd.to_numeric(group[stat], errors="coerce").dropna()
            if values.empty:
                continue
            mean_value = float(values.mean())
            std_value = float(values.std(ddof=0))
            row[f"{stat}_mean"] = mean_value
            row[f"{stat}_std"] = std_value
            row[f"{stat}_cv"] = float(std_value / mean_value) if mean_value > 0 else np.nan
            row[f"{stat}_p20"] = float(values.quantile(0.20))
            row[f"{stat}_p50"] = float(values.quantile(0.50))
            row[f"{stat}_p80"] = float(values.quantile(0.80))
            row[f"{stat}_above_mean_rate"] = float((values > mean_value).mean())
        rows.append(row)

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    for stat in ["pts", "ast", "reb", "min"]:
        cv_col = f"{stat}_cv"
        if cv_col in result.columns:
            result[f"{stat}_volatility_z"] = result.groupby("season")[cv_col].transform(robust_zscore)
            result[f"{stat}_consistency_score"] = (-result[f"{stat}_volatility_z"]).clip(-3, 3)

    volatility_cols = [c for c in ["pts_volatility_z", "ast_volatility_z", "reb_volatility_z", "min_volatility_z"] if c in result.columns]
    result["overall_volatility_score"] = result[volatility_cols].mean(axis=1)
    result["consistency_label"] = pd.Series(
        np.select([result["overall_volatility_score"] <= -0.5, result["overall_volatility_score"] >= 0.5], ["consistent", "volatile"], default="balanced"),
        index=result.index,
    )
    return result.sort_values(["season_start_year", "player_id"]).reset_index(drop=True)


player_consistency_signals = build_player_consistency_signals(performance)
player_consistency_signals.to_parquet(CONSISTENCY_SIGNAL_PATH, index=False)
print(f"Saved {CONSISTENCY_SIGNAL_PATH} with shape {player_consistency_signals.shape}")
display(player_consistency_signals.tail(10))

## Short-Term Expected/Floor/Ceiling Ranges

This section creates a deterministic next-five-game range estimate. It uses only point-in-time rolling features available at row `t` and evaluates against the existing next-five-game targets.

The expected value is a weighted blend of season average, last-10 average, and last-5 average. The floor and ceiling are based on the player's recent rolling volatility.

In [ ]:
def add_recent_rolling_volatility(performance_df: pd.DataFrame, window: int = 10) -> pd.DataFrame:
    """Input: clean performance table. Output: same table with rolling player-season standard deviations through row t."""
    result = performance_df.sort_values(["player_id", "season", "as_of_date", "game_id"]).copy()
    for stat in ["pts", "ast", "reb"]:
        result[f"{stat}_rolling_std_{window}"] = result.groupby(["player_id", "season"])[stat].transform(lambda s: s.rolling(window=window, min_periods=5).std(ddof=0))
    return result


def build_short_term_floor_ceiling_signals(performance_df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean point-in-time performance table. Output: expected, floor, and ceiling ranges for next-five PTS/AST/REB."""
    df = add_recent_rolling_volatility(performance_df).copy()
    output_cols = ["player_id", "as_of_date", "game_id", "season", "team_id", "split"]

    for stat, config in STAT_CONFIG.items():
        expected_col = f"expected_next_5_{stat}_avg"
        floor_col = f"floor_next_5_{stat}_avg"
        ceiling_col = f"ceiling_next_5_{stat}_avg"
        width_col = f"range_width_next_5_{stat}_avg"
        std_col = f"{stat}_rolling_std_10"
        target_col = config["target"]

        df[expected_col] = 0.50 * df[config["season_avg"]] + 0.30 * df[config["last_10"]] + 0.20 * df[config["last_5"]]
        fallback_std = df.groupby("season")[config["actual"]].transform("std")
        volatility = df[std_col].fillna(fallback_std).fillna(0)
        df[floor_col] = (df[expected_col] - 0.80 * volatility).clip(lower=0)
        df[ceiling_col] = df[expected_col] + 0.80 * volatility
        df[width_col] = df[ceiling_col] - df[floor_col]
        output_cols.extend([expected_col, floor_col, ceiling_col, width_col, target_col])

    target_cols = [config["target"] for config in STAT_CONFIG.values()]
    return df[output_cols].dropna(subset=target_cols).reset_index(drop=True)


def evaluate_floor_ceiling(signals: pd.DataFrame) -> pd.DataFrame:
    """Input: floor-ceiling signal table. Output: split/stat metrics for expected values and range coverage."""
    rows = []
    for split_name, split_df in signals.groupby("split"):
        for stat, config in STAT_CONFIG.items():
            target_col = config["target"]
            pred_col = f"expected_next_5_{stat}_avg"
            floor_col = f"floor_next_5_{stat}_avg"
            ceiling_col = f"ceiling_next_5_{stat}_avg"
            valid = split_df[[target_col, pred_col, floor_col, ceiling_col]].dropna()
            if valid.empty:
                continue
            y_true = valid[target_col].to_numpy(dtype="float64")
            y_pred = valid[pred_col].to_numpy(dtype="float64")
            rows.append({
                "split": split_name,
                "stat": stat,
                "rows": len(valid),
                "mae": mean_absolute_error(y_true, y_pred),
                "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
                "r2": r2_score(y_true, y_pred),
                "coverage_rate": ((valid[target_col] >= valid[floor_col]) & (valid[target_col] <= valid[ceiling_col])).mean(),
                "avg_range_width": (valid[ceiling_col] - valid[floor_col]).mean(),
            })
    return pd.DataFrame(rows).sort_values(["stat", "split"]).reset_index(drop=True)


short_term_floor_ceiling_signals = build_short_term_floor_ceiling_signals(performance)
short_term_floor_ceiling_evaluation = evaluate_floor_ceiling(short_term_floor_ceiling_signals)

short_term_floor_ceiling_signals.to_parquet(FLOOR_CEILING_SIGNAL_PATH, index=False)
short_term_floor_ceiling_evaluation.to_parquet(FLOOR_CEILING_EVAL_PATH, index=False)

print(f"Saved {FLOOR_CEILING_SIGNAL_PATH} with shape {short_term_floor_ceiling_signals.shape}")
print(f"Saved {FLOOR_CEILING_EVAL_PATH} with shape {short_term_floor_ceiling_evaluation.shape}")
display(short_term_floor_ceiling_evaluation)
display(short_term_floor_ceiling_signals.tail(10))

## Floor/Ceiling Range Calibration

The expected/floor/ceiling formula has tunable parameters. Instead of treating the weights as fixed assumptions, this section uses a validation optimizer and selects the best configuration on the validation split.

Selection rule:

1. Optimize expected-value weights by minimizing normalized validation MAE across PTS/AST/REB.
2. Optimize the volatility multiplier to reach the validation coverage target.
3. Report calibrated validation/test metrics against the original fixed config.

The selected weights can later be copied into local `RangeConfig` after reviewing the output.

In [ ]:
RANGE_COVERAGE_TARGET = 0.80
STAT_CONFIG_TARGETS = {stat: config["target"] for stat, config in STAT_CONFIG.items()}


def prepare_range_calibration_frame(performance_df: pd.DataFrame) -> pd.DataFrame:
    """Input: performance table. Output: sorted table with rolling volatility columns needed for fast calibration."""
    df = performance_df.sort_values(["player_id", "season", "as_of_date", "game_id"]).copy()
    for stat in STAT_CONFIG:
        std_col = f"{stat}_rolling_std_10"
        if std_col not in df.columns:
            df[std_col] = (
                df.groupby(["player_id", "season"])[stat]
                .transform(lambda s: s.rolling(10, min_periods=5).std(ddof=0))
            )
        fallback_col = f"{stat}_fallback_std"
        df[fallback_col] = df.groupby("season")[STAT_CONFIG[stat]["actual"]].transform("std")
        df[std_col] = df[std_col].fillna(df[fallback_col]).fillna(0)
    return df


def softmax_weights(raw_weights: np.ndarray) -> np.ndarray:
    """Input: unconstrained optimizer weights. Output: positive weights that sum to one."""
    shifted = raw_weights - np.max(raw_weights)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def normalized_expected_mae_for_weights(df: pd.DataFrame, weights: np.ndarray, split_name: str = "validation") -> float:
    """Input: calibration dataframe and three expected weights. Output: average normalized MAE across stats."""
    split_df = df[df["split"].eq(split_name)]
    stat_errors = []
    for stat, stat_config in STAT_CONFIG.items():
        target_col = STAT_CONFIG_TARGETS[stat]
        expected = (
            weights[0] * split_df[stat_config["season_avg"]]
            + weights[1] * split_df[stat_config["last_10"]]
            + weights[2] * split_df[stat_config["last_5"]]
        )
        valid = pd.concat([expected.rename("expected"), split_df[target_col]], axis=1).dropna()
        if valid.empty:
            continue
        scale = valid[target_col].std(ddof=0)
        if pd.isna(scale) or scale == 0:
            scale = 1.0
        stat_errors.append(mean_absolute_error(valid[target_col], valid["expected"]) / scale)
    return float(np.mean(stat_errors))


def calibrate_expected_weights(df: pd.DataFrame, split_name: str = "validation") -> tuple[np.ndarray, float, str]:
    """Input: calibration dataframe. Output: selected expected weights, objective value, and optimizer name."""
    try:
        from scipy.optimize import differential_evolution

        def objective(raw_weights: np.ndarray) -> float:
            return normalized_expected_mae_for_weights(df, softmax_weights(raw_weights), split_name=split_name)

        result = differential_evolution(
            objective,
            bounds=[(-2.0, 2.0), (-2.0, 2.0), (-2.0, 2.0)],
            seed=42,
            maxiter=35,
            popsize=8,
            polish=True,
            workers=1,
        )
        weights = softmax_weights(result.x)
        return weights, float(result.fun), "scipy_differential_evolution"
    except Exception as exc:
        print("SciPy optimization unavailable; falling back to small grid:", exc)
        candidates = []
        for season_weight in [0.30, 0.40, 0.50, 0.60, 0.70]:
            for last_10_weight in [0.10, 0.20, 0.30, 0.40, 0.50]:
                last_5_weight = 1.0 - season_weight - last_10_weight
                if last_5_weight < 0:
                    continue
                weights = np.array([season_weight, last_10_weight, last_5_weight])
                candidates.append((normalized_expected_mae_for_weights(df, weights, split_name=split_name), weights))
        score, weights = sorted(candidates, key=lambda item: item[0])[0]
        return weights, float(score), "fallback_grid_search"


def coverage_for_volatility_multiplier(df: pd.DataFrame, weights: np.ndarray, volatility_multiplier: float, split_name: str = "validation") -> tuple[float, float]:
    """Input: weights and volatility multiplier. Output: average coverage and average range width across stats."""
    split_df = df[df["split"].eq(split_name)]
    coverages = []
    widths = []
    for stat, stat_config in STAT_CONFIG.items():
        target_col = STAT_CONFIG_TARGETS[stat]
        expected = (
            weights[0] * split_df[stat_config["season_avg"]]
            + weights[1] * split_df[stat_config["last_10"]]
            + weights[2] * split_df[stat_config["last_5"]]
        )
        volatility = split_df[f"{stat}_rolling_std_10"]
        floor = (expected - volatility_multiplier * volatility).clip(lower=0)
        ceiling = expected + volatility_multiplier * volatility
        valid = pd.concat([split_df[target_col], floor.rename("floor"), ceiling.rename("ceiling")], axis=1).dropna()
        if valid.empty:
            continue
        coverages.append(((valid[target_col] >= valid["floor"]) & (valid[target_col] <= valid["ceiling"])).mean())
        widths.append((valid["ceiling"] - valid["floor"]).mean())
    return float(np.mean(coverages)), float(np.mean(widths))


def calibrate_volatility_multiplier(df: pd.DataFrame, weights: np.ndarray, split_name: str = "validation", target_coverage: float = RANGE_COVERAGE_TARGET) -> tuple[float, float, float, str]:
    """Input: selected expected weights. Output: volatility multiplier selected by validation coverage target."""
    try:
        from scipy.optimize import minimize_scalar

        def objective(multiplier: float) -> float:
            coverage, width = coverage_for_volatility_multiplier(df, weights, multiplier, split_name=split_name)
            return abs(coverage - target_coverage) + 1e-4 * width

        result = minimize_scalar(objective, bounds=(0.30, 1.80), method="bounded", options={"xatol": 1e-3})
        multiplier = float(result.x)
        coverage, width = coverage_for_volatility_multiplier(df, weights, multiplier, split_name=split_name)
        return multiplier, coverage, width, "scipy_minimize_scalar"
    except Exception as exc:
        print("SciPy scalar optimization unavailable; falling back to grid:", exc)
        candidates = []
        for multiplier in np.arange(0.30, 1.81, 0.05):
            coverage, width = coverage_for_volatility_multiplier(df, weights, float(multiplier), split_name=split_name)
            candidates.append((abs(coverage - target_coverage), width, float(multiplier), coverage))
        _, width, multiplier, coverage = sorted(candidates, key=lambda item: (item[0], item[1]))[0]
        return multiplier, coverage, width, "fallback_grid_search"


def build_short_term_floor_ceiling_signals_with_calibrated_config(performance_df: pd.DataFrame, selected_config: pd.Series) -> pd.DataFrame:
    """Input: performance table and selected config. Output: calibrated expected/floor/ceiling signal table."""
    df = prepare_range_calibration_frame(performance_df)
    output_cols = ["player_id", "as_of_date", "game_id", "season", "team_id", "split"]
    weights = np.array([selected_config["season_avg_weight"], selected_config["last_10_weight"], selected_config["last_5_weight"]], dtype="float64")
    multiplier = float(selected_config["volatility_multiplier"])
    for stat, stat_config in STAT_CONFIG.items():
        expected_col = f"expected_next_5_{stat}_avg"
        floor_col = f"floor_next_5_{stat}_avg"
        ceiling_col = f"ceiling_next_5_{stat}_avg"
        width_col = f"range_width_next_5_{stat}_avg"
        target_col = STAT_CONFIG_TARGETS[stat]
        expected = (
            weights[0] * df[stat_config["season_avg"]]
            + weights[1] * df[stat_config["last_10"]]
            + weights[2] * df[stat_config["last_5"]]
        )
        volatility = df[f"{stat}_rolling_std_10"]
        df[expected_col] = expected
        df[floor_col] = (expected - multiplier * volatility).clip(lower=0)
        df[ceiling_col] = expected + multiplier * volatility
        df[width_col] = df[ceiling_col] - df[floor_col]
        output_cols.extend([expected_col, floor_col, ceiling_col, width_col, target_col])
    return df[output_cols].dropna(subset=list(STAT_CONFIG_TARGETS.values())).reset_index(drop=True)


range_calibration_frame = prepare_range_calibration_frame(performance)
selected_weights, selected_weight_objective, weight_optimizer = calibrate_expected_weights(range_calibration_frame, split_name="validation")
selected_multiplier, selected_coverage, selected_width, multiplier_optimizer = calibrate_volatility_multiplier(range_calibration_frame, selected_weights, split_name="validation")

selected_range_config = pd.DataFrame([
    {
        "season_avg_weight": selected_weights[0],
        "last_10_weight": selected_weights[1],
        "last_5_weight": selected_weights[2],
        "volatility_multiplier": selected_multiplier,
        "validation_normalized_mae": selected_weight_objective,
        "validation_avg_coverage_rate": selected_coverage,
        "validation_avg_range_width": selected_width,
        "weight_optimizer": weight_optimizer,
        "multiplier_optimizer": multiplier_optimizer,
        "selected_by": "validation_normalized_mae_then_validation_coverage_target",
    }
])

calibrated_floor_ceiling_signals = build_short_term_floor_ceiling_signals_with_calibrated_config(performance, selected_range_config.iloc[0])
calibrated_floor_ceiling_evaluation = evaluate_floor_ceiling(calibrated_floor_ceiling_signals)

print("Selected calibrated range config:")
display(selected_range_config)
print("Calibrated split/stat metrics:")
display(calibrated_floor_ceiling_evaluation)

print("Original fixed-config metrics for comparison:")
display(short_term_floor_ceiling_evaluation)

## Replacement Candidate Ranking

Replacement ranking compares player-season role profiles inside the same season. It is deterministic and explainable: candidates are closer when their standardized role/performance vectors are closer, with optional salary and age context for practical scouting filters.

In [ ]:
SIMILARITY_FEATURES = [
    "minutes", "usage_pct", "points_per_100", "assists_per_100", "rebounds_per_100",
    "true_shooting_pct", "three_point_attempt_rate", "free_throw_rate", "turnover_rate",
    "steal_rate", "block_rate", "defensive_rebound_rate", "foul_rate",
    "scoring_creation", "playmaking", "shooting", "rim_pressure", "rebounding",
    "perimeter_defense", "interior_defense", "two_way_impact",
]


def build_similarity_base(role_df: pd.DataFrame, salary_df: pd.DataFrame) -> pd.DataFrame:
    """Input: role features and salary table. Output: player-season rows with role features plus salary context."""
    role = role_df.copy()
    role["season_start_year"] = role["season"].map(season_start_year)
    salary_context = salary_df[["player_id", "season_start_year", "salary_usd", "salary_cap_share", "team_id"]].rename(columns={"team_id": "salary_team_id"})
    return role.merge(salary_context, on=["player_id", "season_start_year"], how="left")


def standardized_similarity_matrix(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """Input: one-season candidate table and feature names. Output: standardized numeric feature matrix and retained features."""
    available = [feature for feature in features if feature in df.columns]
    matrix = df[available].apply(pd.to_numeric, errors="coerce")
    matrix = matrix.fillna(matrix.median(numeric_only=True)).fillna(0)
    scaled = StandardScaler().fit_transform(matrix)
    return pd.DataFrame(scaled, index=df.index, columns=available), available


def find_replacement_candidates(
    base_df: pd.DataFrame,
    player_name: str,
    season: str | None = None,
    top_n: int = 10,
    cheaper_only: bool = False,
    younger_only: bool = False,
    same_position_group: bool = True,
) -> pd.DataFrame:
    """Input: similarity base table and scout filters. Output: ranked replacement candidates for one player-season."""
    base = base_df.copy()
    name_mask = base["player_name"].str.contains(player_name, case=False, na=False, regex=False)
    if season is not None:
        name_mask &= base["season"].eq(season)
    target_rows = base[name_mask].sort_values(["season_start_year", "minutes"], ascending=[False, False])
    if target_rows.empty:
        raise ValueError(f"No player-season row matched player_name={player_name!r}, season={season!r}")

    target = target_rows.iloc[0]
    season_df = base[base["season"].eq(target["season"])].copy()
    season_df = season_df[season_df["player_id"].ne(target["player_id"])]

    if same_position_group and pd.notna(target.get("position")):
        target_position = str(target["position"])
        season_df = season_df[season_df["position"].astype(str).str.contains(target_position[:1], na=False)]

    if cheaper_only and pd.notna(target.get("salary_cap_share")):
        season_df = season_df[season_df["salary_cap_share"].fillna(np.inf) <= float(target["salary_cap_share"])]

    if younger_only and pd.notna(target.get("age")):
        season_df = season_df[season_df["age"].fillna(np.inf) <= float(target["age"])]

    scoring_df = pd.concat([target.to_frame().T, season_df], ignore_index=True)
    matrix, retained_features = standardized_similarity_matrix(scoring_df, SIMILARITY_FEATURES)
    target_vector = matrix.iloc[0].to_numpy(dtype="float64")
    candidate_matrix = matrix.iloc[1:].to_numpy(dtype="float64")
    distances = np.sqrt(((candidate_matrix - target_vector) ** 2).mean(axis=1))

    candidates = scoring_df.iloc[1:].copy()
    candidates["target_player_name"] = target["player_name"]
    candidates["target_player_id"] = target["player_id"]
    candidates["target_season"] = target["season"]
    candidates["similarity_distance"] = distances
    candidates["similarity_score"] = 1 / (1 + candidates["similarity_distance"])
    candidates["salary_cap_share_gap"] = candidates["salary_cap_share"] - target.get("salary_cap_share", np.nan)
    candidates["age_gap"] = candidates["age"] - target.get("age", np.nan)
    candidates["features_used"] = ", ".join(retained_features)

    output_cols = [
        "target_player_name", "target_player_id", "target_season",
        "player_id", "player_name", "season", "team_id", "position", "age",
        "minutes", "usage_pct", "points_per_100", "assists_per_100", "rebounds_per_100",
        "true_shooting_pct", "salary_usd", "salary_cap_share", "salary_cap_share_gap", "age_gap",
        "similarity_distance", "similarity_score", "features_used",
    ]
    return candidates[output_cols].sort_values("similarity_distance").head(top_n).reset_index(drop=True)


similarity_base = build_similarity_base(role_features, salary)
print("similarity_base", similarity_base.shape)

In [ ]:
# Example candidates use recent high-minute players when available. Replace these names in scouting use.
example_requests = [
    {"player_name": "Jayson Tatum", "season": "2024-25", "top_n": 10, "cheaper_only": True, "younger_only": False},
    {"player_name": "Jalen Brunson", "season": "2024-25", "top_n": 10, "cheaper_only": True, "younger_only": False},
    {"player_name": "Bam Adebayo", "season": "2024-25", "top_n": 10, "cheaper_only": True, "younger_only": False},
]

example_tables = []
for request in example_requests:
    try:
        table = find_replacement_candidates(similarity_base, **request)
        table["example_query"] = request["player_name"]
        example_tables.append(table)
    except ValueError as exc:
        print(exc)

replacement_candidate_examples = pd.concat(example_tables, ignore_index=True) if example_tables else pd.DataFrame()
if not replacement_candidate_examples.empty:
    replacement_candidate_examples.to_parquet(REPLACEMENT_EXAMPLES_PATH, index=False)
    print(f"Saved {REPLACEMENT_EXAMPLES_PATH} with shape {replacement_candidate_examples.shape}")
    display(replacement_candidate_examples)
else:
    print("No replacement examples were generated. Update example_requests with players available in the latest season.")

## Heuristic Audit Notes

Calibrated with validation labels:

- Trend thresholds for PTS/AST/REB can be selected from `selected_trend_thresholds`.
- Floor/ceiling expected weights and volatility multiplier can be selected from `selected_range_config`.

Not calibrated in this notebook:

- Consistency labels use observed within-season volatility. They are descriptive signals, not supervised predictions.
- Similarity ranking uses equal-weight standardized role features. Calibrating those weights requires either human relevance labels or a downstream retrieval objective.
- Salary value labels require a trained salary model prediction and should be calibrated after salary model selection.

## Output Summary

The notebook writes deterministic scouting artifacts into the gold layer:

- `player_trend_signals.parquet`
- `player_consistency_signals.parquet`
- `short_term_floor_ceiling_signals.parquet`
- `short_term_floor_ceiling_evaluation.parquet`
- `replacement_candidate_examples.parquet`

These outputs are designed to support scout-facing views and downstream local implementation without adding new supervised model risk.

In [ ]:
output_paths = [
    TREND_SIGNAL_PATH,
    CONSISTENCY_SIGNAL_PATH,
    FLOOR_CEILING_SIGNAL_PATH,
    FLOOR_CEILING_EVAL_PATH,
    REPLACEMENT_EXAMPLES_PATH,
]

for path in output_paths:
    print(path.name, "exists=", path.exists())